# Extracción de Patrimonios Separados (CMF) — Google Colab

Ejecuta el pipeline `securitizadoras/scripts/pipeline_securitizadoras.py` del repo público **monitor-financiero-chile**
y entrega los resultados para integrarlos por PR. No requiere crear tokens ni descargar nada al equipo.

**Uso:** ajusta la celda 3 si quieres otro rango y luego *Entorno de ejecución → Ejecutar todo*.
- Corrida de prueba (`MODO_PRUEBA`): últimos 2 años, sólo diciembre, unos minutos. Recomendada primero.
- Corrida completa (2010→hoy, trimestral, ~3.000 PDFs): varias horas. Colab gratuito corta sesiones inactivas; si
  ocurre, córrela por tramos de 4–5 años (`DESDE`/`HASTA`) y entrega cada tramo.

**Entrega de resultados (celda 6):** se autentica con GitHub por *device flow* (te muestra un código de 8 caracteres y
un enlace; lo apruebas en el navegador con tu cuenta, sin crear tokens) y sube una rama `colab/ps-extraccion-…`.
Si prefieres no autenticar, la celda 7 genera un `.zip` que se descarga solo; adjúntalo en la conversación de Arena.

In [ ]:
#@title 1. Repositorio y dependencias
import os, subprocess
REPO = "joaquinignaciomondaca-code/monitor-financiero-chile"
RAMA_BASE = "arena/01a0dbf4-monitor-financiero-chile"   #@param {type:"string"}
if not os.path.isdir("/content/repo"):
    !git clone -q --branch {RAMA_BASE} --depth 1 https://github.com/{REPO}.git /content/repo
%cd /content/repo
!git config user.name "colab-extraccion" && git config user.email "colab@users.noreply.github.com"
!pip -q install -r requirements.txt 2>&1 | tail -1
!python securitizadoras/tests/test_pipeline_securitizadoras.py | tail -1

In [ ]:
#@title 2. Conectividad con CMF (si falla aquí, CMF bloquea la IP de Colab: corre el mismo comando en tu equipo)
import urllib.request, ssl
try:
    html = urllib.request.urlopen(urllib.request.Request("https://www.cmfchile.cl/institucional/mercados/consulta_busqueda.php?valor=securitizadora", headers={"User-Agent": "Mozilla/5.0"}), timeout=30).read()
    print("OK, CMF responde:", len(html), "bytes;", html.count(b"RGSEC"), "securitizadoras en la búsqueda")
except Exception as e:
    print("SIN ACCESO A CMF desde Colab:", e)
    print("Alternativa: en tu PC → git clone …; pip install -r requirements.txt; python securitizadoras/scripts/pipeline_securitizadoras.py --step todo --desde 2010")

In [ ]:
#@title 3. Parámetros de la extracción
DESDE = 2010        #@param {type:"integer"}
HASTA = 2026        #@param {type:"integer"}
TRIMESTRES = "03,06,09,12"  #@param ["12", "06,12", "03,06,09,12"] {allow-input: true}
MODO_PRUEBA = False #@param {type:"boolean"}
if MODO_PRUEBA:
    DESDE, TRIMESTRES = max(DESDE, HASTA - 2), "12"
print(f"Rango {DESDE}-{HASTA}, trimestres {TRIMESTRES}")

In [ ]:
#@title 4. Ejecutar pipeline (maestro + gestoras + patrimonios separados)
import time; t0 = time.time()
!python securitizadoras/scripts/pipeline_securitizadoras.py --step todo --desde {DESDE} --hasta {HASTA} --trimestres {TRIMESTRES} 2>&1 | tee /content/pipeline.log | grep -v "^  [A-Z].*lineas=" 
!grep -c "lineas=" /content/pipeline.log | xargs -I{} echo "PDFs con líneas extraídas: {}"
print(f"Duración: {(time.time()-t0)/60:.1f} min")

In [ ]:
#@title 5. Auditoría y resumen de cobertura
!python securitizadoras/scripts/audit_patrimonios_separados_v2.py --json /content/audit.json; echo "exit=$?"
import duckdb
c = duckdb.connect()
print(c.sql("select estado, count(*) n from 'docs/outputs/securitizadoras/patrimonios_separados_cobertura.parquet' group by 1 order by 2 desc"))
print(c.sql("select nombre_administradora, count(distinct id_patrimonio) ps, min(periodo) desde, max(periodo) hasta, count(*) balances, round(100.0*sum(cuadre_contable_ok::int)/count(*),1) pct_cuadre from 'docs/outputs/securitizadoras/patrimonios_separados_balance_resumen.parquet' group by 1 order by 1"))
print(c.sql("select glosa, count(*) n from 'docs/outputs/securitizadoras/patrimonios_separados_eeff_lineas.parquet' where cuenta_canonica is null group by 1 order by 2 desc limit 25"))

In [ ]:
#@title 6. Entregar resultados: rama en GitHub (autenticación por navegador, sin tokens)
import datetime, shutil
rama = f"colab/ps-extraccion-{datetime.datetime.utcnow():%Y%m%d-%H%M}"
!git checkout -q -b {rama}
!git add docs/outputs/securitizadoras/
!git commit -q -m "securitizadoras: extracción CMF desde Colab ({DESDE}-{HASTA}, trimestres {TRIMESTRES})" && echo commit ok
if shutil.which("gh") is None:
    !curl -sSL https://github.com/cli/cli/releases/download/v2.63.2/gh_2.63.2_linux_amd64.deb -o /tmp/gh.deb && apt-get -qq install -y /tmp/gh.deb >/dev/null && echo "gh instalado"
# Muestra un código y una URL: ábrela, pega el código y autoriza. Luego vuelve aquí.
!gh auth login --hostname github.com --git-protocol https --web --skip-ssh-key < /dev/null || gh auth status
!gh auth setup-git
!git push -q -u origin {rama} && echo "\nRama subida: {rama}\nPega este nombre de rama en la conversación de Arena."


In [ ]:
#@title 7. Alternativa sin autenticación: zip para adjuntar en Arena
import shutil, datetime
from google.colab import files
z = f"/content/ps_extraccion_{DESDE}_{HASTA}_{datetime.datetime.utcnow():%Y%m%d}"
shutil.make_archive(z, "zip", "docs/outputs/securitizadoras")
shutil.copy("/content/pipeline.log", "/content/")
print("Adjunta en la conversación:", z + ".zip")
files.download(z + ".zip")